# TimesFM 3: check the Transformers port on the released checkpoint

Converts `google/timesfm-3.0-pytorch` with the conversion script of the branch `add-timesfm3`, compares the forecasts with Google's implementation, and prints the values for the integration test.

**Runtime → Run all**, then **File → Save a copy in GitHub** (same repository and branch). CPU is enough.

In [1]:
!pip -q install "git+https://github.com/Abdelaziz03/transformers@add-timesfm3" "timesfm[torch] @ git+https://github.com/google-research/timesfm"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.5.0 which is incompatible.


In [2]:
import json, math, os

import torch
from huggingface_hub import hf_hub_download

REPO = "google/timesfm-3.0-pytorch"


def fetch(name):
    return os.path.join(REPO, name) if os.path.isdir(REPO) else hf_hub_download(REPO, name)


try:
    original_config = json.load(open(fetch("config.json")))
    print("checkpoint reachable")
except Exception as error:
    print(type(error).__name__, error)
    print("If the repository is gated: accept its licence on https://huggingface.co/google/timesfm-3.0-pytorch,")
    print("run `from huggingface_hub import notebook_login; notebook_login()` in a new cell, then run all again.")
    raise

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

checkpoint reachable


In [3]:
from safetensors.torch import load_file

import transformers
from transformers.models.timesfm3.convert_timesfm3_original_to_hf import convert_checkpoint, verify_conversion

print("transformers", transformers.__version__, "| torch", torch.__version__)
model = convert_checkpoint(original_config, load_file(fetch("model.safetensors")))
print("parameters:", sum(p.numel() for p in model.parameters()))

try:
    verify_conversion(model, REPO)
    print("VERIFY: passed")
except ValueError as error:
    print("VERIFY: failed:", error)

transformers 5.25.0.dev0 | torch 2.14.1+cpu


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.32GB            

model.safetensors: downloading bytes:           |  0.00B            

parameters: 330710976
covariates=False, flip invariance=False: largest difference 6.3e-07 of the largest forecast
covariates=False, flip invariance=True: largest difference 3.6e-07 of the largest forecast
covariates=True, flip invariance=False: largest difference 5.4e-07 of the largest forecast
covariates=True, flip invariance=True: largest difference 3.6e-07 of the largest forecast
VERIFY: passed


## How much Google's own two attention paths differ

The same inputs, through Google's code with `use_sdpa=True` and `use_sdpa=False`, and through the port. Differences are relative to the largest forecast.

In [4]:
from timesfm3 import TimesFM3Forecaster

google_sdpa = TimesFM3Forecaster.from_pretrained(REPO, device="cpu", use_sdpa=True)
google_manual = TimesFM3Forecaster.from_pretrained(REPO, device="cpu", use_sdpa=False)

generator = torch.Generator().manual_seed(0)
context_length, prediction_length = 8 * 32 + 5, 4 * 32 + 3
steps = torch.arange(context_length, dtype=torch.float32)
targets = 10 * torch.sin(2 * torch.pi * steps / 24) + torch.randn(2, 2, context_length, generator=generator)
targets[1] += 0.05 * steps
targets[0, :, :3] = float("nan")
targets[0, 0, 15:19] = float("nan")
past_covariates = torch.randn(2, 1, context_length, generator=generator)
known_covariates = torch.randn(2, 1, context_length + prediction_length, generator=generator)


def google(forecaster, with_covariates, flip):
    covariates = {}
    if with_covariates:
        covariates = {
            "past_only_covariates": list(past_covariates.numpy()),
            "past_future_covariates": list(known_covariates.numpy()),
        }
    outputs = forecaster.predict_batch(
        list(targets.numpy()), horizon=prediction_length, return_quantiles=True,
        use_symmetric_averaging=flip, make_positive=True, **covariates,
    )
    return torch.stack([torch.from_numpy(output.quantiles) for output in outputs])


def port(with_covariates, flip):
    covariates = {"past_covariates": past_covariates, "known_covariates": known_covariates} if with_covariates else {}
    with torch.no_grad():
        return model(
            past_values=targets, prediction_length=prediction_length,
            force_flip_invariance=flip, truncate_negative=True, **covariates,
        ).full_predictions


def relative(first, second):
    return ((first - second).abs().max() / second.abs().max()).item()


print("covariates | flip  | largest forecast | Google sdpa vs manual | port vs Google sdpa | port vs Google manual")
for with_covariates in (False, True):
    for flip in (False, True):
        sdpa, manual, ours = google(google_sdpa, with_covariates, flip), google(google_manual, with_covariates, flip), port(with_covariates, flip)
        print(
            f"RESULT {str(with_covariates):5s} | {str(flip):5s} | {sdpa.abs().max().item():.4g} | "
            f"{relative(manual, sdpa):.2e} | {relative(ours, sdpa):.2e} | {relative(ours, manual):.2e}"
        )

covariates | flip  | largest forecast | Google sdpa vs manual | port vs Google sdpa | port vs Google manual
RESULT False | False | 31.23 | 5.50e-07 | 6.26e-07 | 5.80e-07
RESULT False | True  | 31.6 | 3.62e-07 | 3.62e-07 | 3.62e-07
RESULT True  | False | 31.77 | 6.30e-07 | 5.40e-07 | 6.60e-07
RESULT True  | True  | 31.8 | 4.20e-07 | 3.60e-07 | 6.00e-07


## Values for the integration test

In [5]:
steps = torch.arange(512, dtype=torch.float32)
past_values = torch.sin(2 * math.pi * steps / 24)[None, None, :]
with torch.no_grad():
    outputs = model(past_values=past_values, prediction_length=64)
expected = next(iter(google_sdpa.predict_batch([past_values[0].numpy()], horizon=64, return_quantiles=True))).quantiles


def show(values):
    return "[" + ", ".join(f"{value:.6f}" for value in values.tolist()) + "]"


print("INTEGRATION shape", tuple(outputs.full_predictions.shape), "| quantiles", list(model.config.quantiles))
print("INTEGRATION median[:8] ", show(outputs.mean_predictions[0, 0, :8]))
print("INTEGRATION median[-4:]", show(outputs.mean_predictions[0, 0, -4:]))
print("INTEGRATION q10[:4]    ", show(outputs.full_predictions[0, 0, :4, 0]))
print("INTEGRATION q90[:4]    ", show(outputs.full_predictions[0, 0, :4, 8]))
print("INTEGRATION difference with Google on this input:", f"{relative(outputs.full_predictions[0], torch.from_numpy(expected)):.2e}")

INTEGRATION shape (1, 1, 64, 9) | quantiles [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
INTEGRATION median[:8]  [0.864650, 0.703734, 0.498623, 0.258608, 0.001887, -0.256365, -0.496276, -0.704066]
INTEGRATION median[-4:] [-0.865357, -0.708215, -0.503280, -0.262230]
INTEGRATION q10[:4]     [0.861019, 0.700785, 0.495857, 0.252802]
INTEGRATION q90[:4]     [0.868762, 0.708234, 0.501612, 0.260958]
INTEGRATION difference with Google on this input: 1.54e-06


In [6]:
model.save_pretrained("timesfm-3.0-transformers")
reloaded = transformers.TimesFm3ModelForPrediction.from_pretrained("timesfm-3.0-transformers")
with torch.no_grad():
    again = reloaded(past_values=past_values, prediction_length=64).full_predictions
print("SAVED and reloaded; identical forecasts:", torch.equal(again, outputs.full_predictions))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

SAVED and reloaded; identical forecasts: True
